# Module 1.1 — Introduction to RAG

**Retrieval-Augmented Generation (RAG)** combines the power of a retrieval system with a generative language model. Instead of relying solely on knowledge baked into the model weights, RAG fetches relevant documents at inference time and injects them into the prompt.

## Problems RAG Solves
| Problem | How RAG Helps |
|---|---|
| Hallucination | Grounds answers in retrieved facts |
| Knowledge cutoff | Retrieves up-to-date external documents |
| Domain-specific knowledge | Ingest proprietary / niche corpora |
| Source attribution | Every answer traceable to a source chunk |

## Core Components
```
User Query → [Retriever] → Relevant Chunks → [Generator / LLM] → Answer
                 ↑
           Knowledge Base (Vector Store)
```

## 1.1.2 — End-to-End RAG Data Flow

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install langchain langchain-openai langchain-community chromadb openai python-dotenv

import os
from dotenv import load_dotenv
load_dotenv()   # expects OPENAI_API_KEY in a .env file

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.schema import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── 1. Build a tiny knowledge base ───────────────────────────────────────────
docs = [
    Document(page_content="RAG stands for Retrieval-Augmented Generation.",
             metadata={"source": "intro.txt"}),
    Document(page_content="LangChain is a framework for building LLM applications.",
             metadata={"source": "intro.txt"}),
    Document(page_content="Vector stores index embeddings for fast similarity search.",
             metadata={"source": "intro.txt"}),
    Document(page_content="Hallucination is a major problem with LLMs — RAG reduces it.",
             metadata={"source": "intro.txt"}),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="rag_intro")
retriever   = vectorstore.as_retriever(search_kwargs={"k": 2})

# ── 2. Prompt template ────────────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer ONLY using the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
""")

# ── 3. LLM ───────────────────────────────────────────────────────────────────
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── 4. Chain: retrieve → format → prompt → llm → parse ───────────────────────
def format_docs(docs):
    return "\n\n".join(
        f"[Source: {d.metadata.get('source','?')}]\n{d.page_content}" for d in docs
    )

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ── 5. Ask a question ─────────────────────────────────────────────────────────
query  = "What is RAG and why is it useful?"
answer = chain.invoke(query)
print("Question:", query)
print("Answer  :", answer)


## 1.1.3 — Visualising the RAG Pipeline

```
┌─────────────┐    embed     ┌─────────────────┐
│  User Query │─────────────▶│  Vector Store   │
└─────────────┘              │  (Chroma/FAISS) │
                             └────────┬────────┘
                                      │ top-k chunks
                             ┌────────▼────────┐
                             │  Prompt Builder │
                             └────────┬────────┘
                                      │ filled prompt
                             ┌────────▼────────┐
                             │   LLM (GPT-4o)  │
                             └────────┬────────┘
                                      │ answer
                             ┌────────▼────────┐
                             │  Final Response  │
                             └─────────────────┘
```

In [ ]:
# ── Inspect retrieved documents ───────────────────────────────────────────────
retrieved = retriever.invoke(query)
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"  [{i}] {doc.page_content}")
    print(f"       Source: {doc.metadata}\n")
